# Tutorial 19 -- Anharmonicity and Leakage Under Strong Drive

Compare weak and strong resonant driving in a three-level transmon and track the population that leaks into `|f>`.

**Prerequisites.** Tutorial 18 is recommended first.


## 1. Goal

We will demonstrate how stronger resonant `g-e` driving in a multilevel model starts to populate `|f>` even when the carrier is aimed at the lowest transition.


## 2. Physical Background

A real transmon is not an ideal two-level system. When the Rabi rate becomes too large compared with the anharmonicity, the drive can no longer cleanly isolate the `g-e` manifold.


## 3. Imports


In [ ]:
from __future__ import annotations

from functools import partial
from pathlib import Path
import sys

REPO_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "pyproject.toml").exists() and (candidate / "cqed_sim").is_dir()
    ),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Could not resolve the repository root from the notebook working directory.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import qutip as qt

from cqed_sim import (
    AmplifierChain,
    BosonicModeSpec,
    DispersiveCouplingSpec,
    DispersiveReadoutTransmonStorageModel,
    DispersiveTransmonCavityModel,
    DisplacementGate,
    FrameSpec,
    NoiseSpec,
    Pulse,
    PurcellFilter,
    QubitMeasurementSpec,
    ReadoutChain,
    ReadoutResonator,
    RotationGate,
    SidebandDriveSpec,
    SequenceCompiler,
    SimulationConfig,
    StatePreparationSpec,
    TransmonModeSpec,
    UniversalCQEDModel,
    build_displacement_pulse,
    build_rotation_pulse,
    build_sideband_pulse,
    carrier_for_transition_frequency,
    coherent_state,
    compute_energy_spectrum,
    fock_state,
    manifold_transition_frequency,
    measure_qubit,
    prepare_simulation,
    prepare_state,
    pure_dephasing_time_from_t1_t2,
    qubit_state,
    run_rabi,
    run_ramsey,
    run_spectroscopy,
    run_t1,
    run_t2_echo,
    sideband_transition_frequency,
    simulate_batch,
    simulate_sequence,
)
from cqed_sim.plotting import plot_energy_levels
from cqed_sim.pulses import gaussian_envelope, square_envelope
from cqed_sim.sim import (
    cavity_wigner,
    conditioned_bloch_xyz,
    mode_moments,
    qubit_conditioned_mode_moments,
    readout_response_by_qubit_state,
    reduced_cavity_state,
    reduced_qubit_state,
    reduced_storage_state,
    storage_photon_number,
    subsystem_level_population,
    transmon_level_populations,
)
from tutorials.tutorial_support import (
    GHz,
    MHz,
    angular_to_ghz,
    angular_to_hz,
    angular_to_mhz,
    cross_kerr_conditional_phase,
    final_expectation,
    fit_echo_signal,
    fit_exponential_decay,
    fit_lorentzian_peak,
    fit_rabi_vs_amplitude,
    fit_rabi_vs_duration,
    fit_ramsey_signal,
    gaussian_quasistatic_echo_excited_population,
    gaussian_quasistatic_ramsey_excited_population,
    ns,
    ramsey_population,
    resonant_drive_excited_population,
    t1_relaxation_population,
    us,
)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (7.0, 4.2)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False


## 4. Simulation Parameters


In [ ]:
weak_drive = 2.0 * np.pi * 8.0e6
strong_drive = 2.0 * np.pi * 45.0e6
pulse_duration = 80.0 * ns
dt = 1.0 * ns


## 5. Model Construction


In [ ]:
model = DispersiveTransmonCavityModel(
    omega_c=0.0,
    omega_q=GHz(6.2),
    alpha=MHz(-220.0),
    chi=0.0,
    kerr=0.0,
    n_cav=1,
    n_tr=3,
)
frame = FrameSpec(omega_q_frame=model.omega_q)


## 6. Pulse / Sequence Construction


In [ ]:
def run_drive(amplitude):
    pulse = Pulse("q", 0.0, pulse_duration, square_envelope, amp=amplitude, carrier=0.0, label="strong_drive")
    compiled = SequenceCompiler(dt=dt).compile([pulse], t_end=pulse_duration + dt)
    return simulate_sequence(
        model,
        compiled,
        model.basis_state(0, 0),
        {"q": "qubit"},
        config=SimulationConfig(frame=frame, store_states=True, max_step=dt),
    )

weak_result = run_drive(weak_drive)
strong_result = run_drive(strong_drive)


## 7. Running the Simulation


In [ ]:
weak_pf = np.asarray(weak_result.expectations["P_f"], dtype=float)
strong_pf = np.asarray(strong_result.expectations["P_f"], dtype=float)
print(f"Weak-drive maximum P_f = {np.max(weak_pf):.4f}")
print(f"Strong-drive maximum P_f = {np.max(strong_pf):.4f}")


## 8. Visualizing the Results


In [ ]:
time_ns = np.asarray(weak_result.solver_result.times, dtype=float) * 1.0e9
fig, axes = plt.subplots(1, 2, figsize=(12.0, 4.4), sharey=True)
axes[0].plot(time_ns, weak_result.expectations["P_e"], label=r"$P_e$")
axes[0].plot(time_ns, weak_result.expectations["P_f"], label=r"$P_f$")
axes[0].set_title("Weak drive")
axes[0].set_xlabel("Time [ns]")
axes[0].set_ylabel("Population")
axes[0].legend()

axes[1].plot(time_ns, strong_result.expectations["P_e"], label=r"$P_e$")
axes[1].plot(time_ns, strong_result.expectations["P_f"], label=r"$P_f$")
axes[1].set_title("Strong drive")
axes[1].set_xlabel("Time [ns]")
axes[1].legend()
plt.show()


## 9. Physical Interpretation

The strong-drive trace leaks more noticeably into `|f>` because the drive is no longer perturbatively small compared with the transmon anharmonicity. That is the multilevel reason to be careful with large pulse amplitudes.


## 10. Exercises / Next Steps

- Change the anharmonicity and repeat the comparison.
- Try a Gaussian envelope instead of a square pulse to see how the spectral width changes the leakage behavior.
- Continue to Tutorial 20 for another kind of numerical care: truncation convergence.
